# 10 - MobileNetV2 Partial Fine-Tuning

**RUN_E2_TRAINING=False.** Run Sections 1-3 for preparation and disposable GPU verification only. Every model starts from the hash-pinned E1 epoch-10 checkpoint; no pretrained download, previous training rerun, or test-image access.

Privately attach the existing `outputs/models/E1_mobilenetv2_frozen_seed42.pt` retaining its filename, plus the existing extracted `enge707-gpu-setup` input. No extra bundle is needed. The competition input can stay attached but is not read here.

Required working files: `/kaggle/working/enge707_train_cache/` and `/kaggle/working/training_image_cache_check.json`. Kaggle can remove these between sessions. If missing, stop: explicitly restore the verified raw-JPEG cache and report, or deliberately run notebook07's existing cache-preparation cell after its required setup. This notebook never copies images or imports another notebook. In a fresh session it can read the pinned extracted setup directly.

Separate preparation, disposable verification, and guarded training. No GPU/cross-version bitwise reproducibility or exact-resume guarantee.

## 1. Explicit definitions and trainability policy

Restore E1 with weights=None. Independently control E2 train(): selected feature modules and classifier dropout train, earlier modules stay in eval, and all BatchNorm affine parameters and running statistics stay frozen. Unfreeze only non-BatchNorm parameters in modules 16-18 plus the original classifier.

In [ ]:
RUN_E2_TRAINING = False
E2_RUN_COMPLETED = False
from pathlib import Path
import gc, hashlib, importlib, json, platform, random, statistics, struct, sys, time
import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image, __version__ as pillow_version
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import mobilenet_v2

WORKING_ROOT = Path("/kaggle/working")
TRAIN_CACHE_DIR = WORKING_ROOT / "enge707_train_cache"
CACHE_REPORT = WORKING_ROOT / "training_image_cache_check.json"
SETUP_ROOT = (WORKING_ROOT / "enge707_gpu_setup" if (WORKING_ROOT / "enge707_gpu_setup").exists()
              else Path("/kaggle/input/datasets/leocaoprojects/enge707-gpu-setup"))
E1_CHECKPOINT = None  # Optional explicit path if Kaggle renames the attached file.
E1_SHA256 = "4043403fda03a02184038e817395f8001744b8542f99993383e046175b90b968"
E1_CONFIG_SHA256 = "b1be48735a9c4a942a276f2396453f26d8fc2e584ff601c1f57ce1fe2d6141cb"
TRANSFER_SHA256 = "14b63f88302e255b3f2acca0a34377ee35e5dc5fddada8136651d236d36fc8b8"
COUNTS = {"train": 50927, "validation": 12733}
RUN_NAME = "E2_mobilenetv2_finetuned_seed42"
DIAGNOSTIC_PATH = WORKING_ROOT / "mobilenetv2_finetuning_setup_check.json"


def digest(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def require(condition, message):
    if not condition:
        raise ValueError(message)


def seed_all():
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)


def save_json(path, value, exclusive=False):
    with Path(path).open("x" if exclusive else "w", encoding="utf-8") as stream:
        json.dump(value, stream, indent=2, allow_nan=False)
        stream.write("\n")


def runtime():
    return {"python": platform.python_version(), "torch": str(torch.__version__),
            "torchvision": torchvision.__version__, "numpy": np.__version__, "pandas": pd.__version__,
            "Pillow": pillow_version, "cuda_build": torch.version.cuda}

class ImageNetNormalizedDataset(Dataset):
    """Wrap verified /255 tensors; retain IDs, labels, geometry and augmentation."""
    def __init__(self, base):
        self.base = base
        self.mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32)[:, None, None]
        self.std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32)[:, None, None]

    def __len__(self):
        return len(self.base)

    def __getitem__(self, index):
        image, label, image_id = self.base[index]
        if image.dtype != torch.float32 or image.shape != (3, 224, 224):
            raise ValueError("Expected reference-preprocessed float32 [3,224,224]")
        return (image - self.mean) / self.std, label, image_id

class FrozenMobileNetV2(nn.Module):
    """Official pretrained features with an eight-logit, trainable classifier."""
    def __init__(self, weights=None):
        super().__init__()
        if weights is not None and not Path("/kaggle/working").is_dir():
            raise RuntimeError("Pretrained downloads are allowed in Kaggle only")
        self.network = mobilenet_v2(weights=weights)
        self.network.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(self.network.last_channel, 8))
        for parameter in self.network.features.parameters():
            parameter.requires_grad_(False)
        self.train()

    def train(self, mode=True):
        super().train(mode)
        # Frozen weights alone do not freeze BatchNorm running statistics.
        self.network.features.eval()
        return self

    def forward(self, images):
        return self.network(images)

class PartiallyFineTunedMobileNetV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = mobilenet_v2(weights=None)
        self.network.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(self.network.last_channel, 8))
        names = list(dict(self.network.features.named_children()))
        require(names == [str(i) for i in range(19)], "Unexpected torchvision feature-module layout")
        self.selected_names = names[-3:]
        require(self.selected_names == ["16", "17", "18"]
                and [type(self.network.features[int(i)]).__name__ for i in self.selected_names]
                == ["InvertedResidual", "InvertedResidual", "Conv2dNormActivation"], "Final modules changed")
        self.requires_grad_(False)
        for name in self.selected_names:
            self.network.features[int(name)].requires_grad_(True)
        self.network.classifier.requires_grad_(True)
        for module in self.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.requires_grad_(False)
        require(sum(p.numel() for p in self.parameters()) == 2234120
                and sum(p.numel() for p in self.parameters() if p.requires_grad) == 1205128,
                "Architecture/trainable parameter count changed")
        self.train()

    def train(self, mode=True):
        # Independent from E1.train(): selected features must be allowed to train.
        super().train(mode)
        for name, module in self.network.features.named_children():
            module.train(mode and name in self.selected_names)
        for module in self.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()
        return self

    def forward(self, images):
        return self.network(images)  # Same eight raw logits and classifier as E1.


def parameter_groups(model):
    features = [p for p in model.network.features.parameters() if p.requires_grad]
    classifier = [p for p in model.network.classifier.parameters() if p.requires_grad]
    require(sum(p.numel() for p in features) == 1194880 and sum(p.numel() for p in classifier) == 10248,
            "Feature/classifier counts changed")
    require(not set(map(id, features)) & set(map(id, classifier)), "Optimizer groups overlap")
    return [{"params": features, "lr": 1e-5, "name": "features"},
            {"params": classifier, "lr": 1e-4, "name": "classifier"}]


def frozen_snapshot(model):
    names = {name for name, p in model.named_parameters() if not p.requires_grad}
    names.update(name for name, _ in model.named_buffers())
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items() if name in names}


def verify_frozen(model, before):
    state = model.state_dict()
    require(all(torch.equal(state[name].detach().cpu(), value) for name, value in before.items()),
            "Frozen parameters or buffers changed")
    require(all(p.grad is None for p in model.parameters() if not p.requires_grad), "Frozen gradient found")
    require(all(not module.training and all(not p.requires_grad for p in module.parameters(recurse=False))
                for module in model.modules() if isinstance(module, nn.modules.batchnorm._BatchNorm)),
            "BatchNorm policy changed")


def batch_step(model, batch, device, optimizer=None):
    images, targets, ids = batch
    require(images.dtype == torch.float32 and targets.dtype == torch.int64, "Invalid batch dtype")
    require(bool(torch.isfinite(images).all()) and bool(((targets >= 0) & (targets < 8)).all()), "Invalid inputs/labels")
    images, targets = images.to(device), targets.to(device)
    if optimizer is not None:
        optimizer.zero_grad(set_to_none=True)
    with (torch.enable_grad() if optimizer is not None else torch.inference_mode()), torch.autocast(device_type=device.type, enabled=False):
        logits = model(images)
        loss = nn.functional.cross_entropy(logits, targets)  # Unweighted raw-logit CE.
        require(logits.shape == (len(ids), 8) and bool(torch.isfinite(logits).all() & torch.isfinite(loss)),
                "Invalid logits or loss")
        if optimizer is not None:
            loss.backward()
            for group in optimizer.param_groups:
                gradients = [p.grad for p in group["params"]]
                require(all(g is not None and bool(torch.isfinite(g).all()) for g in gradients),
                        "Missing/nonfinite trainable gradients")
                require(any(bool((g != 0).any()) for g in gradients), "All gradients zero in an optimizer group")
            optimizer.step()
    return {"loss_sum": float(loss.detach()) * len(ids),
            "correct": int((logits.detach().argmax(1) == targets).sum()), "samples": len(ids)}


def run_epoch(model, loader, device, optimizer=None):
    model.train(optimizer is not None)
    loss_sum = correct = samples = 0
    seen = []
    expected = loader.dataset.base.rows["image_id"].tolist()
    for batch in loader:
        result = batch_step(model, batch, device, optimizer)
        loss_sum += result["loss_sum"]
        correct += result["correct"]
        samples += result["samples"]
        seen.extend(batch[2])
    require(samples == len(expected) and samples > 0 and len(set(seen)) == samples
            and set(seen) == set(expected), "Incomplete/repeated epoch IDs")
    require(optimizer is not None or seen == expected, "Validation order changed")
    return {"loss": loss_sum / samples, "accuracy": correct / samples, "samples": samples}


## 2. Preparation only

Fail before model construction or image decoding if the checkpoint, setup, or verified cache/report is missing. Verify the pinned manifest and all six payloads before importing the original Dataset. Require the same 12 strict preprocessing fingerprints and unchanged split/mapping. Filename enumeration plus the prior full cache hash report verify cache coverage; this does not repeat the whole-cache byte audit.

In [ ]:
def restore_e1(path, fine_tune=True):
    require(digest(path) == E1_SHA256, "E1 checkpoint SHA256 mismatch")
    checkpoint = torch.load(path, weights_only=True, map_location="cpu")
    canonical = json.dumps(checkpoint["configuration"], sort_keys=True, separators=(",", ":")).encode()
    require(hashlib.sha256(canonical).hexdigest() == E1_CONFIG_SHA256, "E1 configuration changed")
    require(checkpoint["best_epoch"] == 10 and checkpoint["best_validation_loss"] == 0.1460895589143588,
            "Wrong E1 epoch or best loss")
    model = PartiallyFineTunedMobileNetV2() if fine_tune else FrozenMobileNetV2(weights=None)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    return model, checkpoint


def make_loaders(prepared):
    seed_all()
    bases = {name: FacialFeatureDataset(prepared["setup"], name, training_image_directory=TRAIN_CACHE_DIR)
             for name in COUNTS}
    for name, base in bases.items():
        require(len(base) == COUNTS[name] and base.rows.equals(prepared["rows"][name][["image_id", "class", "image_path"]]),
                "Dataset membership/order changed")
    require(bases["validation"].augmentation is None
            and repr(bases["train"].augmentation) == prepared["e1_config"]["training_augmentation"],
            "Original augmentation changed")
    datasets = {name: ImageNetNormalizedDataset(base) for name, base in bases.items()}
    return {"train": DataLoader(datasets["train"], batch_size=32, num_workers=0, drop_last=False,
                               shuffle=True, generator=torch.Generator().manual_seed(42)),
            "validation": DataLoader(datasets["validation"], batch_size=32, num_workers=0, drop_last=False, shuffle=False)}


def verify_inputs(prepared):
    for name, path in prepared["input_paths"].items():
        require(digest(path) == prepared["input_sha256"][name], "Input changed: " + name)


def prepare_e2():
    require(WORKING_ROOT.is_dir(), "Kaggle /kaggle/working required; do not run preparation locally")
    require(TRAIN_CACHE_DIR.is_dir() and not TRAIN_CACHE_DIR.is_symlink() and CACHE_REPORT.is_file(),
            "Missing verified cache/report: restore /kaggle/working/enge707_train_cache and "
            "training_image_cache_check.json, or explicitly run notebook07's cache-preparation cell "
            "after its required verified setup. No automatic dataset copy is performed.")
    require(SETUP_ROOT.is_dir() and not SETUP_ROOT.is_symlink(),
            "Attach the existing extracted enge707-gpu-setup input or restore the verified working setup.")
    if E1_CHECKPOINT is not None:
        candidates = [Path(E1_CHECKPOINT)]
    else:
        candidates = list(Path("/kaggle/input").rglob("E1_mobilenetv2_frozen_seed42.pt"))
        hosted = WORKING_ROOT / "E1_mobilenetv2_frozen_seed42/best_checkpoint.pt"
        if hosted.is_file():
            candidates.append(hosted)
    require(len(candidates) == 1 and candidates[0].is_file() and not candidates[0].is_symlink(),
            "Attach exactly one E1_mobilenetv2_frozen_seed42.pt privately, or set E1_CHECKPOINT explicitly.")
    checkpoint_path = candidates[0].resolve()
    require(digest(checkpoint_path) == E1_SHA256, "Transferred E1 checkpoint differs from local original")
    manifest_path = SETUP_ROOT / "transfer_manifest.json"
    require(digest(manifest_path) == TRANSFER_SHA256, "Pinned transfer manifest mismatch")
    transfer = json.loads(manifest_path.read_text())
    expected_payload = {"src/__init__.py", "src/data_pipeline.py", "src/models.py", "data/processed/split_manifest.csv",
                        "data/processed/class_mapping.json", "preprocessing_fingerprints.json"}
    require(set(transfer["sha256"]) == expected_payload, "Unexpected transfer payload")
    input_paths = {name: SETUP_ROOT / name for name in expected_payload}
    for name, path in input_paths.items():
        require(not path.is_symlink() and path.resolve().is_relative_to(SETUP_ROOT.resolve())
                and not any(parent.is_symlink() for parent in path.parents), "Unsafe payload path")
        require(digest(path) == transfer["sha256"][name], "Payload checksum mismatch: " + name)
    input_paths.update(transfer_manifest=manifest_path, e1_checkpoint=checkpoint_path, cache_report=CACHE_REPORT)
    split = pd.read_csv(input_paths["data/processed/split_manifest.csv"], dtype=str, keep_default_na=False, na_filter=False)
    mapping = json.loads(input_paths["data/processed/class_mapping.json"].read_text())
    names = sorted(mapping["class_to_index"])
    require(len(names) == 8 and mapping == transfer["classes"] and mapping["class_to_index"] == dict(zip(names, range(8)))
            and all(mapping["index_to_class"][str(i)] == name for i, name in enumerate(names)), "Mapping changed")
    require(len(split) == 63660 and split["image_id"].is_unique and split["image_id"].str.fullmatch(r"[0-9]{5}").all()
            and split["split"].value_counts().to_dict() == COUNTS
            and split["image_path"].equals("data/raw/train/train/" + split["image_id"] + ".jpg"), "Split contract changed")
    rows = {name: split.loc[split["split"].eq(name)].reset_index(drop=True) for name in COUNTS}
    overlaps = {key: len(set(rows["train"][key]) & set(rows["validation"][key]))
                for key in ["image_id", "byte_sha256", "pixel_sha256"]}
    require(not any(overlaps.values()) and all(set(frame["class"]) == set(names) for frame in rows.values()),
            "Split overlap or class coverage failure")
    cache_report = json.loads(CACHE_REPORT.read_text())
    require(cache_report.get("status") == "passed" and cache_report.get("verified_files") == 63660
            and cache_report.get("hash_checks") == 63660 and not cache_report.get("failures")
            and not cache_report.get("hash_mismatches") and cache_report.get("checks")
            and all(cache_report["checks"].values())
            and Path(cache_report["cache_directory"]).resolve() == TRAIN_CACHE_DIR.resolve(), "Incomplete cache report")
    files = list(TRAIN_CACHE_DIR.iterdir())
    require(len(files) == 63660 and {path.name for path in files} == set(split["image_id"] + ".jpg")
            and all(path.is_file() and not path.is_symlink() and path.resolve().parent == TRAIN_CACHE_DIR.resolve()
                    for path in files), "Missing/extra/unsafe cache files")
    # Import original source only after all payload bytes pass verification.
    for name, module in list(sys.modules.items()):
        if (name == "src" or name.startswith("src.")) and getattr(module, "__file__", None):
            require(Path(module.__file__).resolve().is_relative_to(SETUP_ROOT.resolve()),
                    "Different src code already imported; use a clean verified session")
    if str(SETUP_ROOT.resolve()) not in sys.path:
        sys.path.insert(0, str(SETUP_ROOT.resolve()))
    global FacialFeatureDataset
    pipeline = importlib.import_module("src.data_pipeline")
    FacialFeatureDataset = pipeline.FacialFeatureDataset
    fingerprints = json.loads(input_paths["preprocessing_fingerprints.json"].read_text())["examples"]
    require(len(fingerprints) == len({item["image_id"] for item in fingerprints}) == 12, "Fingerprint set changed")
    for item in fingerprints:
        row = split.loc[split["image_id"].eq(item["image_id"])].iloc[0]
        require(row["split"] == "train" and row["class"] == item["class"], "Fingerprint membership changed")
        path = TRAIN_CACHE_DIR / (item["image_id"] + ".jpg")
        require(digest(path) == item["original_byte_sha256"] == row["byte_sha256"], "Fingerprint JPEG changed")
        with Image.open(path) as image:
            image.load()
            tensor = pipeline.reference_preprocess(image)
        header = b"ENGE707-reference-f32-v1\0" + struct.pack(">3Q", *tensor.shape)
        actual = hashlib.sha256(header + np.asarray(tensor.numpy(), dtype="<f4").tobytes(order="C")).hexdigest()
        require(list(tensor.shape) == [3, 224, 224] and tensor.dtype == torch.float32
                and actual == item["tensor_sha256"], "Preprocessing fingerprint mismatch; do not relax across versions")
    seed_all()
    model, checkpoint = restore_e1(checkpoint_path)
    config = checkpoint["configuration"]
    require(checkpoint["class_mapping"] == mapping == config["class_mapping"]
            and checkpoint["split_checksum"] == digest(input_paths["data/processed/split_manifest.csv"])
            and config["counts"] == COUNTS
            and all(config["input_sha256"][name] == transfer["sha256"][name] for name in expected_payload),
            "E1 metadata differs from verified setup")
    print("Selected modules:", ["network.features." + name for name in model.selected_names])
    print("Total/trainable parameters:", sum(p.numel() for p in model.parameters()),
          sum(p.numel() for p in model.parameters() if p.requires_grad))
    print("LRs: features=1e-5; classifier=1e-4; all BatchNorm parameters/buffers frozen")
    prepared = {"setup": SETUP_ROOT, "checkpoint": checkpoint_path, "mapping": mapping, "rows": rows,
                "e1_config": config, "overlap_counts": overlaps, "input_paths": input_paths,
                "input_sha256": {name: digest(path) for name, path in input_paths.items()}}
    parameter_groups(model)
    del model, checkpoint
    loaders = make_loaders(prepared)
    raw, label, image_id = loaders["validation"].dataset.base[0]
    normalised, repeated_label, repeated_id = loaders["validation"].dataset[0]
    dataset = loaders["validation"].dataset
    require(torch.equal(normalised, (raw - dataset.mean) / dataset.std)
            and image_id == repeated_id and label == repeated_label, "Normalisation changed")
    verify_inputs(prepared)
    print("Preparation passed. Next: Section 3 only. Full E2 training is disabled.")
    return prepared

preparation = prepare_e2()


## 3. Disposable GPU verification only

Compare two fixed, unaugmented validation batches before updates against an independent original E1 model. Require eight finite logits matching with `atol=1e-6, rtol=1e-5`; save actual maximum differences. Then run exactly two warm-up training batches, three timed training batches and two timed validation batches on the disposable E2 model. Timings include batch loading, transfer, forward/loss and training backward/Adam, with CUDA synchronisation at boundaries.

Check a 120-second budget between batches; incomplete counts fail the diagnostic and block training. One batch may exceed the budget. Verify updates in each selected feature module and classifier, with every frozen parameter and buffer unchanged. Record GPU peak allocated/reserved memory. An existing diagnostic is protected. Discard diagnostic weights and optimizer state. Full-epoch projections are short-sample estimates, not measured epochs.

In [ ]:
def verify_e2_gpu(prepared):
    require(not DIAGNOSTIC_PATH.exists(), "Diagnostic already exists; preserve it before a deliberate rerun")
    result = {"status": "running", "diagnostic_only": True, "checks": {}, "failures": [],
              "runtime": runtime(), "input_sha256": prepared["input_sha256"],
              "configuration": {"seed": 42, "batch_size": 32, "num_workers": 0, "drop_last": False,
                                "device": "cuda:0", "dtype": "float32", "amp": False,
                                "feature_lr": 1e-5, "classifier_lr": 1e-4},
              "batch_seconds": {"warmup": [], "training": [], "validation": []},
              "comparison_atol": 1e-6, "comparison_rtol": 1e-5, "comparison_max_abs_differences": [],
              "limitations": ["Rough epoch projection is a short-sample estimate, not a measured epoch.",
                              "Timing includes loading, transfer, verification and cache effects.",
                              "No GPU/cross-version bitwise reproducibility or exact-resume guarantee."]}
    model = baseline = optimizer = loaders = batch = images = reference = actual = None
    train_iterator = val_iterator = iterator = None
    try:
        verify_inputs(prepared)
        require(torch.cuda.is_available(), "CUDA unavailable; enable one Kaggle GPU and use cuda:0")
        device = torch.device("cuda:0")
        result["device"] = {"selected": str(device), "name": torch.cuda.get_device_name(0),
                            "free_bytes": torch.cuda.mem_get_info(0)[0], "cpu_threads": torch.get_num_threads()}
        seed_all()
        model, _ = restore_e1(prepared["checkpoint"])
        baseline, _ = restore_e1(prepared["checkpoint"], fine_tune=False)
        model, baseline = model.to(device=device, dtype=torch.float32), baseline.to(device=device, dtype=torch.float32)
        model.eval(); baseline.eval()
        loaders = make_loaders(prepared)
        iterator = iter(loaders["validation"])
        with torch.inference_mode():
            for number in range(2):
                batch = next(iterator)
                require(list(batch[2]) == prepared["rows"]["validation"]["image_id"].iloc[number*32:(number+1)*32].tolist(),
                        "Fixed comparison validation order changed")
                images = batch[0].to(device)
                reference, actual = baseline(images), model(images)
                require(actual.shape == (32, 8) and bool(torch.isfinite(actual).all() & torch.isfinite(reference).all()),
                        "Invalid comparison logits")
                result["comparison_max_abs_differences"].append(float((actual - reference).abs().max()))
                require(torch.allclose(actual, reference, atol=1e-6, rtol=1e-5), "Restored E1 equivalence failed")
        result["checks"]["two_fixed_validation_batches_match_original_e1"] = True
        baseline = None
        frozen = frozen_snapshot(model)
        before = {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}
        optimizer = torch.optim.Adam(parameter_groups(model))
        model.train()
        require(model.network.classifier[0].training
                and all(model.network.features[int(name)].training for name in model.selected_names),
                "Selected features/classifier dropout did not enter train mode")
        verify_frozen(model, frozen)
        result["selected_modules"] = ["network.features." + name for name in model.selected_names]
        result["parameters"] = {"total": sum(p.numel() for p in model.parameters()),
                                "trainable": sum(p.numel() for p in model.parameters() if p.requires_grad)}
        loaders = make_loaders(prepared)
        labels_by_id = {name: rows.set_index("image_id")["class"].map(prepared["mapping"]["class_to_index"])
                        for name, rows in prepared["rows"].items()}
        train_iterator, val_iterator = iter(loaders["train"]), iter(loaders["validation"])
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)
        started = time.perf_counter()
        for phase, count in [("warmup", 2), ("training", 3), ("validation", 2)]:
            model.train(phase != "validation")
            for number in range(count):
                require(time.perf_counter() - started < 120, "120s diagnostic budget exhausted; no training enabled")
                torch.cuda.synchronize(device)
                tick = time.perf_counter()
                batch = next(val_iterator if phase == "validation" else train_iterator)
                partition = "validation" if phase == "validation" else "train"
                require(batch[1].tolist() == labels_by_id[partition].loc[list(batch[2])].tolist(), "Batch labels changed")
                batch_step(model, batch, device, None if phase == "validation" else optimizer)
                torch.cuda.synchronize(device)
                result["batch_seconds"][phase].append(time.perf_counter() - tick)
                print(phase, number + 1, f"{result['batch_seconds'][phase][-1]:.3f}s", flush=True)
        verify_frozen(model, frozen)
        changed = {name for name, p in model.named_parameters() if name in before
                   and not torch.equal(p.detach().cpu(), before[name])}
        require(any(name.startswith("network.classifier.") for name in changed), "Classifier did not update")
        for name in model.selected_names:
            require(any(key.startswith("network.features." + name + ".") for key in changed), "Selected module did not update: " + name)
        verify_inputs(prepared)
        result["checks"].update(finite_loss_and_gradients=True, selected_feature_layers_updated=True,
            classifier_updated=True, frozen_parameters_and_all_buffers_unchanged=True,
            train_modes_and_bn_policy=True, original_inputs_unchanged=True)
        result["completed_batches"] = {phase: len(times) for phase, times in result["batch_seconds"].items()}
        result["mean_batch_seconds"] = {phase: statistics.mean(times) for phase, times in result["batch_seconds"].items()}
        result["loader_batch_counts"] = {name: len(loader) for name, loader in loaders.items()}
        result["rough_epoch_estimate_seconds"] = (result["mean_batch_seconds"]["training"] * len(loaders["train"])
                                                  + result["mean_batch_seconds"]["validation"] * len(loaders["validation"]))
        result["peak_gpu_allocated_bytes"] = torch.cuda.max_memory_allocated(device)
        result["peak_gpu_reserved_bytes"] = torch.cuda.max_memory_reserved(device)
        result["timing_phase_seconds"] = time.perf_counter() - started
        result["status"] = "passed"
    except Exception as error:
        result["status"] = "failed"
        result["failures"].append(f"{type(error).__name__}: {error}")
        raise
    finally:
        model = baseline = optimizer = loaders = batch = images = reference = actual = None
        train_iterator = val_iterator = iterator = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        result["completed_batches"] = {phase: len(times) for phase, times in result["batch_seconds"].items()}
        result["mean_batch_seconds"] = {phase: statistics.mean(times) if times else None
                                         for phase, times in result["batch_seconds"].items()}
        result["diagnostic_model_and_optimizer_discarded"] = True
        save_json(DIAGNOSTIC_PATH, result, exclusive=True)
        print("Diagnostic:", result["status"], DIAGNOSTIC_PATH)
    print("Rough epoch estimate (NOT measured):", result["rough_epoch_estimate_seconds"] / 60, "minutes")
    print("Full E2 training remains disabled.")
    return result

finetuning_setup_check = verify_e2_gpu(preparation)


## 4. Guarded E2 training - disabled

Do not enable yet. Later, require successful live/on-disk GPU diagnostics, unchanged inputs and the same runtime. Reserve the new E2 directory exclusively; never resume or overwrite an existing run. Reset seeds and restore a fresh model from the original E1 checkpoint, with fresh loaders and Adam state.

Maximum 8 epochs, early stopping after 3 consecutive epochs without strictly lower validation loss, ReduceLROnPlateau factor 0.5/patience 2/absolute threshold 0. Keep sample-weighted loss/accuracy and full epoch coverage, including partial batches. Log both learning rates before and after scheduling. Save configuration, history after each epoch, best checkpoint by validation loss and completion status. No E0/E1 files are written.

In [ ]:
RUN_E2_TRAINING = False
E2_RUN_COMPLETED = False


def train_e2(prepared, diagnostic):
    require(DIAGNOSTIC_PATH.is_file(), "Run GPU verification first")
    saved = json.loads(DIAGNOSTIC_PATH.read_text())
    require(saved == diagnostic and saved["status"] == "passed" and not saved["failures"]
            and saved["checks"] and all(saved["checks"].values())
            and saved["completed_batches"] == {"warmup": 2, "training": 3, "validation": 2}
            and saved["diagnostic_model_and_optimizer_discarded"]
            and saved["input_sha256"] == prepared["input_sha256"]
            and saved["runtime"] == runtime(), "Successful unchanged hosted diagnostic required")
    require(torch.cuda.is_available() and torch.cuda.get_device_name(0) == saved["device"]["name"], "GPU changed")
    verify_inputs(prepared)
    run = WORKING_ROOT / RUN_NAME
    require(not run.exists(), "E2 output directory exists; no overwrite or automatic resume")
    device = torch.device("cuda:0")
    seed_all()
    loaders = make_loaders(prepared)
    model, checkpoint = restore_e1(prepared["checkpoint"])
    model = model.to(device=device, dtype=torch.float32)
    frozen = frozen_snapshot(model)
    optimizer = torch.optim.Adam(parameter_groups(model))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5,
        patience=2, threshold=0, threshold_mode="abs")
    configuration = {"run_name": RUN_NAME, "seed": 42, "max_epochs": 8, "early_stopping_patience": 3,
        "optimizer": "Adam", "initial_learning_rates": {"features": 1e-5, "classifier": 1e-4},
        "loss": "unweighted CrossEntropyLoss on raw logits", "batch_size": 32, "num_workers": 0,
        "drop_last": False, "train_shuffle": True, "validation_shuffle": False,
        "device": "cuda:0", "dtype": "float32", "amp": False, "multi_gpu": False,
        "scheduler": {"name": "ReduceLROnPlateau", "factor": 0.5, "patience": 2, "threshold": 0, "threshold_mode": "abs"},
        "selected_modules": saved["selected_modules"], "parameters": saved["parameters"],
        "trainability_policy": "Final three feature modules' non-BatchNorm parameters and classifier train; all BN parameters and buffers frozen",
        "architecture": prepared["e1_config"]["architecture"], "class_mapping": prepared["mapping"], "counts": COUNTS,
        "preprocessing": prepared["e1_config"]["preprocessing"],
        "training_augmentation": prepared["e1_config"]["training_augmentation"], "validation_augmentation": None,
        "training_image_directory": str(TRAIN_CACHE_DIR), "input_sha256": prepared["input_sha256"],
        "split_sha256": prepared["input_sha256"]["data/processed/split_manifest.csv"],
        "e1_checkpoint": {"path": str(prepared["checkpoint"]), "sha256": E1_SHA256, "best_epoch": 10,
                          "best_validation_loss": checkpoint["best_validation_loss"], "original_weights": prepared["e1_config"]["weights"]},
        "diagnostic_sha256": digest(DIAGNOSTIC_PATH), "runtime": runtime(), "gpu": saved["device"],
        "best_checkpoint_rule": "lowest validation loss", "exact_resume_supported": False}
    run.mkdir(exist_ok=False)
    save_json(run / "configuration.json", configuration, exclusive=True)
    completion = {"status": "running", "completed_epochs": 0, "failures": []}
    save_json(run / "completion_status.json", completion, exclusive=True)
    history = []
    best_loss, best_epoch, stale = float("inf"), None, 0
    torch.cuda.synchronize(device)
    started = time.perf_counter()
    try:
        for epoch in range(1, 9):
            torch.cuda.synchronize(device)
            tick = time.perf_counter()
            rates = {group["name"]: float(group["lr"]) for group in optimizer.param_groups}
            training = run_epoch(model, loaders["train"], device, optimizer)
            validation = run_epoch(model, loaders["validation"], device)
            require(training["samples"] == 50927 and validation["samples"] == 12733, "Wrong epoch sample totals")
            verify_frozen(model, frozen)
            if validation["loss"] < best_loss:
                best_loss, best_epoch, stale = validation["loss"], epoch, 0
                torch.save({"model_state_dict": model.state_dict(), "class_mapping": prepared["mapping"],
                            "configuration": configuration, "split_checksum": configuration["split_sha256"],
                            "best_epoch": best_epoch, "best_validation_loss": best_loss}, run / "best_checkpoint.pt")
            else:
                stale += 1
            scheduler.step(validation["loss"])
            torch.cuda.synchronize(device)
            row = {"epoch": epoch, "training_loss": training["loss"], "training_accuracy": training["accuracy"],
                   "validation_loss": validation["loss"], "validation_accuracy": validation["accuracy"],
                   "training_samples": training["samples"], "validation_samples": validation["samples"],
                   "feature_learning_rate": rates["features"], "classifier_learning_rate": rates["classifier"],
                   "next_feature_learning_rate": optimizer.param_groups[0]["lr"],
                   "next_classifier_learning_rate": optimizer.param_groups[1]["lr"],
                   "epoch_elapsed_seconds": time.perf_counter() - tick, "elapsed_seconds": time.perf_counter() - started}
            history.append(row)
            pd.DataFrame(history).to_csv(run / "history.csv", index=False)
            print(json.dumps(row), flush=True)
            completion.update(completed_epochs=epoch, best_epoch=best_epoch, best_validation_loss=best_loss,
                              elapsed_seconds=row["elapsed_seconds"])
            save_json(run / "completion_status.json", completion)
            if stale >= 3:
                break
        verify_frozen(model, frozen)
        verify_inputs(prepared)
        completion.update(status="completed", frozen_parameters_and_bn_buffers_unchanged=True,
                          history_sha256=digest(run / "history.csv"),
                          stop_reason="early stopping" if stale >= 3 else "maximum epochs")
    except Exception as error:
        completion.update(status="failed", failures=[f"{type(error).__name__}: {error}"])
        raise
    finally:
        save_json(run / "completion_status.json", completion)
        del model, optimizer
        gc.collect()
        torch.cuda.empty_cache()
    print("E2 completed:", completion)
    return completion

if RUN_E2_TRAINING:
    e2_completion = train_e2(preparation, finetuning_setup_check)
    E2_RUN_COMPLETED = True
else:
    print("RUN_E2_TRAINING=False: full training and experiment outputs skipped.")


## 5. Guarded learning curves

Plot only actual, successfully completed E2 history, never diagnostic results. Protect existing curves.

In [ ]:
if RUN_E2_TRAINING:
    require(E2_RUN_COMPLETED and e2_completion["status"] == "completed", "Completed E2 run required")
    run = WORKING_ROOT / RUN_NAME
    completion = json.loads((run / "completion_status.json").read_text())
    require(completion == e2_completion and digest(run / "history.csv") == completion["history_sha256"],
            "Completed history changed")
    history = pd.read_csv(run / "history.csv")
    require(history["epoch"].tolist() == list(range(1, completion["completed_epochs"] + 1))
            and history["training_samples"].eq(50927).all() and history["validation_samples"].eq(12733).all(),
            "Invalid E2 history")
    figure_path = run / "loss_accuracy_curves.png"
    require(not figure_path.exists(), "Curves already exist; no overwrite")
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
    for ax, metric in zip(axes, ["loss", "accuracy"]):
        ax.plot(history["epoch"], history["training_" + metric], label="Training")
        ax.plot(history["epoch"], history["validation_" + metric], label="Validation")
        ax.set(xlabel="Epoch", ylabel=metric.capitalize(), title="E2 " + metric)
        ax.legend()
    with figure_path.open("xb") as stream:
        fig.savefig(stream, format="png", dpi=300)
    plt.show()
    plt.close(fig)
    print("Saved:", figure_path)
else:
    print("RUN_E2_TRAINING=False: learning curves skipped.")
